In [24]:
import pandas as pd
import ast
from sklearn.model_selection import GroupKFold
import numpy as np
import os
import re

In [303]:
df = pd.read_csv('./data/annotated/full_ner/all_annotations_minimal_fixed_multi.csv')

In [304]:
# Replace with your actual folder path
folder_path = "./data/annotated"

excluded_ids = []

# Loop through all .txt files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                excluded_ids.append(line)

# Optional: make sure IDs are unique
excluded_ids = list(set(excluded_ids))

print("Excluded IDs collected from all files:")
print(len(excluded_ids))

Excluded IDs collected from all files:
87


In [306]:
# First, extract the base ID (e.g., 'My_pdf931' from 'My_pdf931_title^abstract')
df['base_doc_id'] = df['doc_id'].apply(lambda x: x.split('_')[0] + '_' + x.split('_')[1] if '_' in x else x)

# Then filter out rows where base_doc_id is in excluded_ids
filtered_df = df[~df['base_doc_id'].isin(excluded_ids)].reset_index(drop=True)

print(f"Filtered DataFrame: {len(df)} → {len(filtered_df)} rows")

Filtered DataFrame: 1614 → 1440 rows


In [307]:
df = filtered_df.copy()

In [309]:
len(set(df['doc_id']))

1440

In [312]:
df['tokens'] = df['tokens'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['ner_tags'] = df['ner_tags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['annotations_tuples'] = df['annotations_tuples'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df['tokens_length'] = df['tokens'].apply(len)
df['ner_tags_length'] = df['ner_tags'].apply(len)
df['length_match'] = df['tokens_length'] == df['ner_tags_length']
df.head()

,doc_id,Text,tokens,ner_tags,annotations_tuples,base_doc_id,tokens_length,ner_tags_length,length_match
0,My_pdf931_title^abstract,Pioglitazone attenuates mitochondrial dysfunct...,"[Pioglitazone, attenuates, mitochondrial, dysf...","[B-therapy-drug, O, O, O, O, O, O, O, O, O, O,...","[(0, 12, therapy-drug, Pioglitazone), (122, 14...",My_pdf931,396,396,True
1,My_pdf931_method,\r\n6 Materials and Methods\r\n6.1 Controlled ...,"[\r\n, 6, Materials, and, Methods, \r\n, 6.1, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, B-welf...","[(72, 169, welfare, Studies were approved by t...",My_pdf931,1453,1453,True
2,My_pdf69new_title^abstract,Inhibition of Factor XIa Reduces the Frequency...,"[Inhibition, of, Factor, XIa, Reduces, the, Fr...","[O, O, O, O, O, O, O, O, O, O, O, O, O, B-dise...","[(93, 120, disease, Carotid Arterial Thrombosi...",My_pdf69new,336,336,True
3,My_pdf69new_method,\r\nMaterials and Methods\r\nAnimals. All anim...,"[\r\n, Materials, and, Methods, \r\n, Animals,...","[O, O, O, O, O, O, O, B-age|B-weight, I-age|I-...","[(34, 219, age, All animal studies were perfor...",My_pdf69new,1996,1996,True
4,My_pdf764_title^abstract,Resveratrol inhibits paclitaxel-induced neurop...,"[Resveratrol, inhibits, paclitaxel, -, induced...","[B-therapy-drug, O, O, O, O, B-disease, I-dise...","[(0, 11, therapy-drug, Resveratrol), (40, 56, ...",My_pdf764,456,456,True


## Split text into sentences

In [408]:
import re

def should_merge(prev, next_):
    """
    Determines whether to merge the current sentence-ending token with the next token,
    based on citation patterns, abbreviations, numbers, and common edge cases in scientific writing.
    
    Parameters:
    - prev (str): The previous token (typically a sentence-ending punctuation like ".")
    - next_ (str): The next token (typically the first token of the potential next sentence)
    
    Returns:
    - bool: True if the tokens should be merged (i.e., no sentence split), False otherwise
    """
    # Handle split medical abbreviations like 'i.c.v' + '.'
    # Case when the abbreviation and the period are in two separate tokens
    if prev.strip().lower() in ['i.c.v', 'i.p', 'i.v', 'i.m', 's.c', 'i.t', 'p.o', 'no', 'al']:
        return True

    # Previous token patterns (e.g., abbreviations or known "no split" cases)
    prev_patterns = [
        r'^(?:i\.c\.v|i\.p|i\.v|i\.m|s\.c|i\.t|p\.o)\.$',              # Administration routes
        r'\b(?:Fig|Figs|Table|Ref|Inc|inc|c|p|g|no|No|vs|sp|spp)\.$',  # Abbreviations
        r'et al\.$',                                                   # "et al."
        r'(?:\s|\.)[A-Z]\.$',                                          # Initials
        r'\)$',                                                        # Closing parenthesis
    ]
    
    # Next token patterns (e.g., signs the sentence should continue)
    next_patterns = [
        r'^,$',                   # Comma (e.g., in citations)
        r'^\)$',                  # Closing parenthesis
        r'^[a-z]',                # Lowercase continuation
        r'^\(',                   # Opening parenthesis
        r'^Mb\b',                 # Scientific abbreviations
        r'^M\b',
        r'^\d',                   # Starts with digit (e.g., year, dose)
        r'^al$',                  # Part of "et al"
        r'^al\.$',
        r'^et$',                  # "et"
        r'^\w+\,?\s?\d{4}[a-z]?$',# Citation year like "2000b"
        r'^\r\n$',                # Newline token
        r'^[A-Z]{2,}[0-9]*$',     # License codes like SCXK
    ]
    
    for pattern in prev_patterns:
        if re.fullmatch(pattern, prev):
            return True

    for pattern in next_patterns:
        if re.fullmatch(pattern, next_):
            return True

    return False


def split_into_sentences(tokens, ner_tags):
    sentences = []
    current_tokens = []
    current_tags = []

    for i in range(len(tokens)):
        tok = tokens[i]
        tag = ner_tags[i]
        current_tokens.append(tok)
        current_tags.append(tag)

        # Check for sentence-ending punctuation
        if tok in [".", "!", "?", "\n"]:
            prev_tok = tokens[i - 1] if i > 0 else ""
            next_tok = tokens[i + 1] if i + 1 < len(tokens) else ""

            # NEW: pass both prev and next into should_merge
            if should_merge(prev_tok, next_tok):
                continue

            # Commit the sentence split
            sentences.append((current_tokens, current_tags))
            current_tokens = []
            current_tags = []

    # Add final remaining chunk
    if current_tokens:
        sentences.append((current_tokens, current_tags))

    return sentences


# Assuming `df` is your DataFrame
new_rows = []

for _, row in df.iterrows():
    sentences = split_into_sentences(row['tokens'], row['ner_tags'])
    for idx, (sent_tokens, sent_tags) in enumerate(sentences):
        new_rows.append({
            'doc_id': row['doc_id'],
            'sentence_id': idx,
            'tokens': sent_tokens,
            'ner_tags': sent_tags,
            'sent_txt': ' '.join(sent_tokens)
        })

sentence_df = pd.DataFrame(new_rows)
sentence_df

,doc_id,sentence_id,tokens,ner_tags,sent_txt
0,My_pdf931_title^abstract,0,"[Pioglitazone, attenuates, mitochondrial, dysf...","[B-therapy-drug, O, O, O, O, O, O, O, O, O, O,...",Pioglitazone attenuates mitochondrial dysfunct...
1,My_pdf931_title^abstract,1,"[Previous, evidence, has, shown, that, activat...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Previous evidence has shown that activation of...
2,My_pdf931_title^abstract,2,"[In, the, current, study, we, hypothesized, th...","[O, O, O, O, O, O, O, O, O, O, B-therapy-drug,...",In the current study we hypothesized that trea...
3,My_pdf931_title^abstract,3,"[Animals, received, a, unilateral, 1.5, mm, co...","[B-model, I-model, I-model, I-model, I-model, ...",Animals received a unilateral 1.5 mm controlle...
4,My_pdf931_title^abstract,4,"[Beginning, 1day, after, the, injury, there, w...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Beginning 1day after the injury there was sign...
...,...,...,...,...,...
38103,My_pdf726_method,33,"[A, positive, pixel, count, algorithm, was, op...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",A positive pixel count algorithm was optimized...
38104,My_pdf726_method,34,"[22, \r\n, Statistical, analysis, \r\n, To, co...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",22 \r\n Statistical analysis \r\n To compare N...
38105,My_pdf726_method,35,"[Generalized, regression, data, analysis, was,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O]",Generalized regression data analysis was used ...
38106,My_pdf726_method,36,"[Mixed, model, regression, data, analysis, and...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Mixed model regression data analysis and Tukey...


In [410]:
def extract_entity_types(tags):
    entity_types = set()
    for tag in tags:
        if tag != "O":
            # Strip B- or I- prefix to get the entity type
            types_list = tag.split("|")
            for types in types_list:
                entity_type = types.split("-", 1)[-1]
                entity_types.add(entity_type)
    return list(entity_types)  # Or use sorted(entity_types) if you want consistency

# Apply this function to your sentence-level dataframe
sentence_df['entity_types'] = sentence_df['ner_tags'].apply(extract_entity_types)
sentence_df

,doc_id,sentence_id,tokens,ner_tags,sent_txt,entity_types
0,My_pdf931_title^abstract,0,"[Pioglitazone, attenuates, mitochondrial, dysf...","[B-therapy-drug, O, O, O, O, O, O, O, O, O, O,...",Pioglitazone attenuates mitochondrial dysfunct...,"[disease, therapy-drug]"
1,My_pdf931_title^abstract,1,"[Previous, evidence, has, shown, that, activat...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Previous evidence has shown that activation of...,[disease]
2,My_pdf931_title^abstract,2,"[In, the, current, study, we, hypothesized, th...","[O, O, O, O, O, O, O, O, O, O, B-therapy-drug,...",In the current study we hypothesized that trea...,"[disease, therapy-drug]"
3,My_pdf931_title^abstract,3,"[Animals, received, a, unilateral, 1.5, mm, co...","[B-model, I-model, I-model, I-model, I-model, ...",Animals received a unilateral 1.5 mm controlle...,"[model, therapy-drug]"
4,My_pdf931_title^abstract,4,"[Beginning, 1day, after, the, injury, there, w...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Beginning 1day after the injury there was sign...,[therapy-drug]
...,...,...,...,...,...,...
38103,My_pdf726_method,33,"[A, positive, pixel, count, algorithm, was, op...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",A positive pixel count algorithm was optimized...,[]
38104,My_pdf726_method,34,"[22, \r\n, Statistical, analysis, \r\n, To, co...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",22 \r\n Statistical analysis \r\n To compare N...,[]
38105,My_pdf726_method,35,"[Generalized, regression, data, analysis, was,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O]",Generalized regression data analysis was used ...,[]
38106,My_pdf726_method,36,"[Mixed, model, regression, data, analysis, and...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Mixed model regression data analysis and Tukey...,[]


In [412]:
sentence_df.to_csv("./data/sentence_split.csv")

In [414]:
sentence_df['age'] = sentence_df['entity_types'].apply(lambda ents: 1 if 'age' in ents else 0)
sentence_df.head()

,doc_id,sentence_id,tokens,ner_tags,sent_txt,entity_types,age
0,My_pdf931_title^abstract,0,"[Pioglitazone, attenuates, mitochondrial, dysf...","[B-therapy-drug, O, O, O, O, O, O, O, O, O, O,...",Pioglitazone attenuates mitochondrial dysfunct...,"[disease, therapy-drug]",0
1,My_pdf931_title^abstract,1,"[Previous, evidence, has, shown, that, activat...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Previous evidence has shown that activation of...,[disease],0
2,My_pdf931_title^abstract,2,"[In, the, current, study, we, hypothesized, th...","[O, O, O, O, O, O, O, O, O, O, B-therapy-drug,...",In the current study we hypothesized that trea...,"[disease, therapy-drug]",0
3,My_pdf931_title^abstract,3,"[Animals, received, a, unilateral, 1.5, mm, co...","[B-model, I-model, I-model, I-model, I-model, ...",Animals received a unilateral 1.5 mm controlle...,"[model, therapy-drug]",0
4,My_pdf931_title^abstract,4,"[Beginning, 1day, after, the, injury, there, w...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Beginning 1day after the injury there was sign...,[therapy-drug],0


In [416]:
sentence_df.age.sum()

478

In [472]:
sentence_df.shape

(38108, 8)

## Pattern based relevant sentences classification

In [426]:
def evaluate_keyword_detection(df, target_col='age', keyword_col='has_age_kw'):
    """
    Computes and prints TP, FP, FN, precision, and recall for keyword-based detection.

    Parameters:
    - df: DataFrame containing the data
    - target_col: column with ground truth labels (0 or 1)
    - keyword_col: column with keyword match results (True/False)

    Returns:
    - precision: float
    - recall: float
    """
    tp = ((df[keyword_col] == True) & (df[target_col] == 1)).sum()
    fp = ((df[keyword_col] == True) & (df[target_col] == 0)).sum()
    fn = ((df[keyword_col] == False) & (df[target_col] == 1)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    print(f"--- Evaluation for '{keyword_col}' predicting '{target_col}' ---")
    print(f"True Positives (TP): {tp}")
    print(f"False Positives (FP): {fp}")
    print(f"False Negatives (FN): {fn}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print("------------------------------------------------------------")

    return precision, recall

In [468]:
# Define the pattern
age_keywords_pattern = (
    r'\b('
    r'age|ages|aged|aging|old|older|young|adult|adults|mature|senescent|'
    r'neonatal|neonate|newborn|pup|pups|juvenile|weanling|weaning|'
    r'postnatal|prenatal|prepubescent|'
    r'fetal|fetus|fetuses|'
    r'day-old|week-old|month-old|year-old|'
    r'\d+[\s-]?(day|week|month|year)s? old'
    r')\b'
)
# Create the boolean mask
sentence_df['has_age_kw'] = sentence_df['sent_txt'].str.contains(age_keywords_pattern, case=False, regex=True)

print("Rows containing age keywords:", sentence_df.has_age_kw.sum())

/var/folders/nd/2fzvhsh510gbt9x6z5pdb1gr0000gn/T/ipykernel_70735/1856922556.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  sentence_df['has_age_kw'] = sentence_df['sent_txt'].str.contains(age_keywords_pattern, case=False, regex=True)


Rows containing 'age', 'aged', 'old', or 'adult': 1507


In [470]:
evaluate_keyword_detection(sentence_df)

--- Evaluation for 'has_age_kw' predicting 'age' ---
True Positives (TP): 443
False Positives (FP): 1064
False Negatives (FN): 35
Precision: 0.29
Recall: 0.93
------------------------------------------------------------


(0.29396151293961514, 0.9267782426778243)

In [478]:
# Define the species patterns
species_patterns = {
    "rat": [r"\brats?\b"],
    "mouse": [r"\bmouse\b", r"\bmice\b"],
    "cat": [r"\bcats?\b"],
    "dog": [r"\bdogs?\b"],
    "guinea pig": [r"\bguinea pig\s?\b", r"\bguinea\b"],
    "monkey": [r"\bmonkeys?\b", r"\bmacaques?\b", r"\bchimpanzees?\b", r"\borangutans?\b", r"\bbonoboss?\b", r"\bgibbons?\b"],
    "pig": [r"\bpigs?\b", r"\bswines?\b", r"\bpiglets?\b"],
    "rabbit": [r"\brabbits?\b"],
    "sheep": [r"\bsheep?\b"],
    "species-other": []  # Fallback
}

# Function to match ent_text to species
def match_species(text):
    for species, patterns in species_patterns.items():
        for pattern in patterns:
            if re.search(pattern, text, flags=re.IGNORECASE):
                return species
    return "species-other"

# Apply to a column like new_df_age['ent_text']
sentence_df['species'] = sentence_df['sent_txt'].apply(match_species)
sentence_df['has_species_kw'] = (sentence_df['species'] != "species-other")

In [480]:
sentence_df

,doc_id,sentence_id,tokens,ner_tags,sent_txt,entity_types,age,has_age_kw,species,has_species_kw
0,My_pdf931_title^abstract,0,"[Pioglitazone, attenuates, mitochondrial, dysf...","[B-therapy-drug, O, O, O, O, O, O, O, O, O, O,...",Pioglitazone attenuates mitochondrial dysfunct...,"[disease, therapy-drug]",0,False,species-other,False
1,My_pdf931_title^abstract,1,"[Previous, evidence, has, shown, that, activat...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Previous evidence has shown that activation of...,[disease],0,False,species-other,False
2,My_pdf931_title^abstract,2,"[In, the, current, study, we, hypothesized, th...","[O, O, O, O, O, O, O, O, O, O, B-therapy-drug,...",In the current study we hypothesized that trea...,"[disease, therapy-drug]",0,False,rat,True
3,My_pdf931_title^abstract,3,"[Animals, received, a, unilateral, 1.5, mm, co...","[B-model, I-model, I-model, I-model, I-model, ...",Animals received a unilateral 1.5 mm controlle...,"[model, therapy-drug]",0,False,species-other,False
4,My_pdf931_title^abstract,4,"[Beginning, 1day, after, the, injury, there, w...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Beginning 1day after the injury there was sign...,[therapy-drug],0,False,species-other,False
...,...,...,...,...,...,...,...,...,...,...
38103,My_pdf726_method,33,"[A, positive, pixel, count, algorithm, was, op...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",A positive pixel count algorithm was optimized...,[],0,False,species-other,False
38104,My_pdf726_method,34,"[22, \r\n, Statistical, analysis, \r\n, To, co...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",22 \r\n Statistical analysis \r\n To compare N...,[],0,False,species-other,False
38105,My_pdf726_method,35,"[Generalized, regression, data, analysis, was,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O]",Generalized regression data analysis was used ...,[],0,False,species-other,False
38106,My_pdf726_method,36,"[Mixed, model, regression, data, analysis, and...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",Mixed model regression data analysis and Tukey...,[],0,False,species-other,False


In [484]:
print("Rows containing species keywords:", sentence_df.has_species_kw.sum())

Rows containing species keywords: 9854


In [482]:
evaluate_keyword_detection(sentence_df, target_col='age', keyword_col='has_species_kw')

--- Evaluation for 'has_species_kw' predicting 'age' ---
True Positives (TP): 447
False Positives (FP): 9407
False Negatives (FN): 31
Precision: 0.05
Recall: 0.94
------------------------------------------------------------


(0.04536228942561396, 0.9351464435146444)

In [486]:
sentence_df[(sentence_df['age'] == 1) & (sentence_df['has_age_kw'] == False)].to_csv("debug_age.csv")
sentence_df[(sentence_df['age'] == 1) & (sentence_df['has_species_kw'] == False)].to_csv("debug_age_species_kw.csv")


In [490]:
len(sentence_df[(sentence_df['age'] == 1) & (sentence_df['has_species_kw'] == False) & (sentence_df['has_age_kw'] == True)])

22

In [79]:
new_df_age.to_csv("data/age_docs.csv")

In [75]:
new_df_age['species'].value_counts()


species
mouse            215
rat              207
species-other     40
sheep              4
dog                4
rabbit             3
monkey             3
pig                2
cat                1
guinea pig         1
Name: count, dtype: int64

In [77]:
new_df_age[new_df_age['species']=="species-other"]

,doc_id,doc_id_unique,ent_type,ent_text,s_idx,e_idx,species,has_age_kw
5,My_pdf484new_method,My_pdf484new_method_1068_1157,age,"For all experiments described, larvae from 0 t...",1068,1157,species-other,False
76,My_pdf385new_method,My_pdf385new_method_851_925,age,Behavioral testing started when animals reache...,851,925,species-other,True
78,My_pdf385new_method,My_pdf385new_method_1392_1440,age,"For Experiment 4, aged (23-months-old) were used",1392,1440,species-other,True
79,My_pdf385new_method,My_pdf385new_method_1630_1873,age,All experimental procedures were performed in ...,1630,1873,species-other,False
116,My_pdf357new_method,My_pdf357new_method_698_741,age,They were used between 6 and 8 weeks of age,698,741,species-other,True
123,My_pdf291new_method,My_pdf291new_method_1813_1992,age,In older animals equivalent methods were used ...,1813,1992,species-other,True
124,My_pdf291new_method,My_pdf291new_method_3741_3907,age,Dexamethasone (0.1 mg/kg) or vehicle was admin...,3741,3907,species-other,True
187,My_pdf118new_method,My_pdf118new_method_383_502,age,"The animals were 3 or 4 years old, weighed bet...",383,502,species-other,True
242,My_pdf206new_method,My_pdf206new_method_443_562,age,"At the time of the experiment, the young adult...",443,562,species-other,True
299,My_pdf823_method,My_pdf823_method_394_500,age,"Male 10-week-old SHRs (SHR/Hos, SPF) were purc...",394,500,species-other,True


## Filter for Drug and Disease

In [22]:
df_disease_drug = df.copy()

In [24]:
# Define the allowed NER tags
allowed_tags = ['B-therapy-drug', 'I-therapy-drug', 'B-disease', 'I-disease']

# Replace all ner_tags not in allowed_tags with 'O'
df_disease_drug['ner_tags'] = df_disease_drug['ner_tags'].apply(lambda tags: ['O' if tag not in allowed_tags else tag for tag in tags])

df_disease_drug['annotations_tuples'] = df_disease_drug['annotations_tuples'].apply(lambda x: eval(x) if isinstance(x, str) else x)

# Filter annotations_tuples to keep only entries with allowed tags
def filter_annotations_tuples(annotations, allowed_tags):
    return [annotation for annotation in annotations if annotation[2] in ['therapy-drug', 'disease']]

df_disease_drug['annotations_tuples'] = df_disease_drug['annotations_tuples'].apply(lambda annots: filter_annotations_tuples(annots, allowed_tags))


In [69]:
df_disease_drug.head()

,doc_id,Text,tokens,ner_tags,annotations_tuples,tokens_length,ner_tags_length,length_match
0,My_pdf931_title^abstract,Pioglitazone attenuates mitochondrial dysfunct...,"[Pioglitazone, attenuates, mitochondrial, dysf...","[B-therapy-drug, O, O, O, O, O, O, O, O, O, O,...","[(0, 12, therapy-drug, Pioglitazone), (122, 14...",396,396,True
1,My_pdf931_method,\r\n6 Materials and Methods\r\n6.1 Controlled ...,"[\r\n, 6, Materials, and, Methods, \r\n, 6.1, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(1310, 1322, therapy-drug, Pioglitazone), (13...",1453,1453,True
2,My_pdf69new_title^abstract,Inhibition of Factor XIa Reduces the Frequency...,"[Inhibition, of, Factor, XIa, Reduces, the, Fr...","[O, O, O, O, O, O, O, O, O, O, O, O, O, B-dise...","[(93, 120, disease, Carotid Arterial Thrombosi...",336,336,True
3,My_pdf69new_method,\r\nMaterials and Methods\r\nAnimals. All anim...,"[\r\n, Materials, and, Methods, \r\n, Animals,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(3936, 3950, therapy-drug, FXIa inhibitor), (...",1996,1996,True
4,My_pdf764_title^abstract,Resveratrol inhibits paclitaxel-induced neurop...,"[Resveratrol, inhibits, paclitaxel, -, induced...","[B-therapy-drug, O, O, O, O, B-disease, I-dise...","[(0, 11, therapy-drug, Resveratrol), (40, 56, ...",456,456,True


In [70]:
def replace_tag(tag):
    if 'therapy-drug' in tag:
        return tag.replace('therapy-drug', 'DRUG')
    elif 'disease' in tag:
        return tag.replace('disease', 'DISEASE')
    return tag

# Apply the replacement to the ner_tags column
df_disease_drug['ner_tags'] = df_disease_drug['ner_tags'].apply(lambda tags: [replace_tag(tag) for tag in tags])

def replace_tag_in_tuple(annotation):
    if isinstance(annotation, tuple) and len(annotation) > 2:
        # Replace 'therapy-drug' or 'disease' in the third element of the tuple (tag)
        tag = annotation[2]
        new_tag = replace_tag(tag)  # Use the existing replace_tag function
        # Return a new tuple with the replaced tag
        return (annotation[0], annotation[1], new_tag, annotation[3])
    return annotation  # If not a valid tuple, return it unchanged

# Apply the replacement to the annotations_tuples column
df_disease_drug['annotations_tuples'] = df_disease_drug['annotations_tuples'].apply(lambda tags: [replace_tag_in_tuple(tag) for tag in tags])

In [74]:
def count_entities(ner_tags):
    b_drug_count = ner_tags.count('B-DRUG')
    b_disease_count = ner_tags.count('B-DISEASE')
    return b_drug_count, b_disease_count

# Apply the function to each row and create new columns for counts
df_disease_drug['B-DRUG_count'], df_disease_drug['B-DISEASE_count'] = zip(*df_disease_drug['ner_tags'].apply(count_entities))


In [75]:
df_disease_drug.head()

,doc_id,Text,tokens,ner_tags,annotations_tuples,tokens_length,ner_tags_length,length_match,B-DRUG_count,B-DISEASE_count
0,My_pdf931_title^abstract,Pioglitazone attenuates mitochondrial dysfunct...,"[Pioglitazone, attenuates, mitochondrial, dysf...","[B-DRUG, O, O, O, O, O, O, O, O, O, O, O, O, O...","[(0, 12, DRUG, Pioglitazone), (122, 144, DISEA...",396,396,True,15,6
1,My_pdf931_method,\r\n6 Materials and Methods\r\n6.1 Controlled ...,"[\r\n, 6, Materials, and, Methods, \r\n, 6.1, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(1310, 1322, DRUG, Pioglitazone), (1342, 1350...",1453,1453,True,10,0
2,My_pdf69new_title^abstract,Inhibition of Factor XIa Reduces the Frequency...,"[Inhibition, of, Factor, XIa, Reduces, the, Fr...","[O, O, O, O, O, O, O, O, O, O, O, O, O, B-DISE...","[(93, 120, DISEASE, Carotid Arterial Thrombosi...",336,336,True,1,4
3,My_pdf69new_method,\r\nMaterials and Methods\r\nAnimals. All anim...,"[\r\n, Materials, and, Methods, \r\n, Animals,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(3936, 3950, DRUG, FXIa inhibitor), (5193, 52...",1996,1996,True,2,0
4,My_pdf764_title^abstract,Resveratrol inhibits paclitaxel-induced neurop...,"[Resveratrol, inhibits, paclitaxel, -, induced...","[B-DRUG, O, O, O, O, B-DISEASE, I-DISEASE, O, ...","[(0, 11, DRUG, Resveratrol), (40, 56, DISEASE,...",456,456,True,11,4


## Cross-Valid Splits

In [77]:
save_loc = "./data/finetuning_ner/finetuning_ner/drug_disease/k_fold_splits/"
# Step 1: Extract the common part of doc_id for grouping (before '_method' or '_title^abstract')
df_disease_drug['group_key'] = df_disease_drug['doc_id'].apply(lambda x: x.split('_')[1])

# Step 2: Prepare GroupKFold for cross-validation
n_splits = 10  # Adjust the number of splits as needed
group_kfold = GroupKFold(n_splits=n_splits)

# Step 3: Split the data using the group_key to ensure rows with the same PDF stay together
# Here, X can be tokens, ner_tags, etc. and y would be ner_tags if this is the target variable
X = df_disease_drug['tokens']  # Features (could be combined features like tokens and ner_tags)
y = df_disease_drug['ner_tags']  # Target (ner_tags)
groups = df_disease_drug['group_key']  # Grouping key

In [120]:
def is_valid_json(obj):
    """Check if an object can be serialized to JSON."""
    try:
        json.dumps(obj)  # Attempt to serialize
        return True
    except (TypeError, ValueError):
        return False

save_full = True
full_data = []  # Store full dataset for final JSON

# Ensure save location exists
os.makedirs(save_loc, exist_ok=True)

# Perform the GroupKFold split
for fold, (train_idx, test_idx) in enumerate(group_kfold.split(X, y, groups)):
    print(f"Fold {fold}:")
    print(f"  Train indices: {len(train_idx)}")
    print(f"  Test indices: {len(test_idx)}")
    
    # Extract train and test sets
    train_data = df_disease_drug.iloc[train_idx]
    test_data = df_disease_drug.iloc[test_idx]
    
    # Save to CSV
    train_data.to_csv(f'{save_loc}train_fold_{fold}.csv', index=False)
    test_data.to_csv(f'{save_loc}test_fold_{fold}.csv', index=False)
    
    # Prepare data for JSON format
    train_json = train_data[['tokens', 'Text', 'ner_tags', 'doc_id']].to_dict(orient='records')
    test_json = test_data[['tokens', 'Text', 'ner_tags', 'doc_id']].to_dict(orient='records')

    # Save each fold as a **JSON array**
    with open(f'{save_loc}train_fold_{fold}.json', 'w', encoding='utf-8') as train_json_file:
        json.dump(train_json, train_json_file, indent=4)  # Pretty JSON format

    with open(f'{save_loc}test_fold_{fold}.json', 'w', encoding='utf-8') as test_json_file:
        json.dump(test_json, test_json_file, indent=4)

    # Add valid data to full dataset
    full_data.extend([row for row in train_json if is_valid_json(row)])
    full_data.extend([row for row in test_json if is_valid_json(row)])

# Save the full dataset as a single JSON array
if save_full:
    output_file = './finetuning_ner/drug_disease/full_ds_drug_disease_ner.json'
    
    with open(output_file, 'w', encoding='utf-8') as json_file:
        json.dump(full_data, json_file, indent=4)  # Save as a single valid JSON array

    print(f"✅ Full dataset saved as JSON array: {output_file}")

Fold 0:
  Train indices: 1419
  Test indices: 158
Fold 1:
  Train indices: 1420
  Test indices: 157
Fold 2:
  Train indices: 1419
  Test indices: 158
Fold 3:
  Train indices: 1419
  Test indices: 158
Fold 4:
  Train indices: 1419
  Test indices: 158
Fold 5:
  Train indices: 1419
  Test indices: 158
Fold 6:
  Train indices: 1419
  Test indices: 158
Fold 7:
  Train indices: 1419
  Test indices: 158
Fold 8:
  Train indices: 1419
  Test indices: 158
Fold 9:
  Train indices: 1421
  Test indices: 156
✅ Full dataset saved as JSON array: ./finetuning_ner/drug_disease/full_ds_drug_disease_ner.json


In [83]:
with open('./finetuning_ner/drug_disease/full_ds_drug_disease_ner.json', 'r', encoding='utf-8') as json_file:
    data = [json.loads(line) for line in json_file]  # Read each line as a JSON object

# Convert the list of dictionaries into a DataFrame
df_loaded = pd.DataFrame(data)

In [99]:
input_file = './finetuning_ner/drug_disease/full_ds_drug_disease_ner.json'
valid_data = []

# Read and filter out bad JSON lines
with open(input_file, 'r', encoding='utf-8') as json_file:
    for line in json_file:
        try:
            entry = json.loads(line.strip())  # Parse JSON line
            valid_data.append(entry)  # Store only valid entries
        except json.JSONDecodeError as e:
            print(f"Skipping corrupted JSON entry: {e}")  # Log error

# Convert to DataFrame
df_loaded = pd.DataFrame(valid_data)

In [111]:
output_file = './finetuning_ner/drug_disease/full_ds_drug_disease_ner_cleaned_array.json'

# Save entire dataset as a JSON array (formatted)
with open(output_file, 'w', encoding='utf-8') as json_file:
    json.dump(valid_data, json_file, indent=4)  # Pretty format with indentation

In [113]:
import json

file_path = './finetuning_ner/drug_disease/full_ds_drug_disease_ner_cleaned_array.json'

try:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("JSON file is valid!")
except json.JSONDecodeError as e:
    print(f"JSON Decode Error: {e}")


JSON file is valid!
